# ESE 5420: Baseline Checkpoint (corrected from project.ipynb)

This notebook retains the original team's Logistic Regression, C grid **[0.01, 0.1, 1, 10]**, stratified 80/20 split (seed 42), and stratified 5-fold CV (seed 42). It corrects two problems in `project.ipynb`: it separates the **five-feature baseline** from **23-feature history comparison**, and it **does not compute test-set scores**. Numeric scaling and one-hot encoding occur inside CV folds.

Put `default of credit card clients.xls` in the same directory as this notebook. Run top to bottom. The 20% holdout is not scored here. **Important:** the earlier team notebook had already evaluated its holdout; that prior exposure cannot be undone and should be disclosed when choosing a final evaluation plan.


In [ ]:
"""Minimal stdlib BIFF8 .xls reader for numeric UCI credit-card file.
Avoids converting the original input or depending on spreadsheet libraries.
Supports OLE2 Workbook stream and NUMBER/RK/MULRK/FORMULA records.
"""
import struct
from collections import defaultdict

END = 0xFFFFFFFE
FREE = 0xFFFFFFFF

def read_workbook_stream(path):
    with open(path,'rb') as f:
        blob=f.read()
    assert blob[:8] == bytes.fromhex('D0CF11E0A1B11AE1'), 'Expected OLE2 xls file'
    u16=lambda pos: struct.unpack_from('<H',blob,pos)[0]
    u32=lambda pos: struct.unpack_from('<I',blob,pos)[0]
    sector_bytes=1<<u16(30)
    assert sector_bytes in (512,4096)
    def sector(sid):
        s=(sid+1)*sector_bytes
        return blob[s:s+sector_bytes]
    num_fat=u32(44)
    fat_ids=list(struct.unpack_from('<109I',blob,76))
    difat_sid=u32(68)
    while difat_sid != END and difat_sid != FREE:
        data=sector(difat_sid)
        ids=struct.unpack_from(f'<{sector_bytes//4}I',data)
        fat_ids.extend(ids[:-1])
        difat_sid=ids[-1]
    fat_ids=[sid for sid in fat_ids if sid not in (FREE,END)][:num_fat]
    FAT=[]
    for sid in fat_ids:
        FAT.extend(struct.unpack_from(f'<{sector_bytes//4}I',sector(sid)))
    def follow(start, max_bytes=None):
        out=bytearray()
        sid=start
        seen=set()
        while sid not in (END, FREE):
            if sid in seen: raise RuntimeError('Cycle in OLE FAT')
            if sid>=len(FAT): raise RuntimeError('Invalid FAT pointer')
            seen.add(sid)
            out.extend(sector(sid))
            sid=FAT[sid]
            if max_bytes is not None and len(out)>=max_bytes: break
        return bytes(out[:max_bytes] if max_bytes else out)
    dirs=follow(u32(48))
    for pos in range(0,len(dirs),128):
        ent=dirs[pos:pos+128]
        if len(ent)<128: break
        name_len=struct.unpack_from('<H',ent,64)[0]
        if name_len<2 or name_len>64: continue
        name=ent[:name_len-2].decode('utf-16le',errors='replace')
        if name.lower() in ('workbook','book'):
            start=struct.unpack_from('<I',ent,116)[0]
            size=struct.unpack_from('<Q',ent,120)[0]
            assert size>=4096, 'Workbook stream unexpectedly tiny'
            return follow(start,size)
    raise ValueError('Workbook not found')

def rk_value(bits):
    # bit0: value divided by 100, bit1: integer rather than floating
    if bits & 2:
        v=struct.unpack('<i',struct.pack('<I',bits & 0xFFFFFFFC))[0] >> 2
    else:
        v=struct.unpack('<d',struct.pack('<Q',(bits & 0xFFFFFFFC)<<32))[0]
    return v/100 if bits & 1 else v

def read_numeric_rows(path, sheet_index=0):
    raw=read_workbook_stream(path)
    off=0
    bounds=[]
    while off+4<=len(raw):
        code, n=struct.unpack_from('<HH',raw,off)
        data=raw[off+4:off+4+n]
        if code == 0x0085 and len(data)>=6:  # BOUNDSHEET
            sheet_off=struct.unpack_from('<I',data,0)[0]
            bounds.append(sheet_off)
        if code==0x000A: break
        off+=4+n
    off=bounds[sheet_index]
    rows=defaultdict(dict)
    record_types=defaultdict(int)
    while off+4<=len(raw):
        code,n=struct.unpack_from('<HH',raw,off)
        d=raw[off+4:off+4+n]
        record_types[hex(code)]+=1
        if code == 0x0203 and n >= 14: # NUMBER
            row,col,xf,val=struct.unpack_from('<HHHd',d)
            rows[row][col]=val
        elif code==0x027E and n>=10: # RK
            row,col,xf,rk=struct.unpack_from('<HHHI',d)
            rows[row][col]=rk_value(rk)
        elif code==0x00BD and n>=12: # MULRK
            row,first_col=struct.unpack_from('<HH',d)
            last_col=struct.unpack_from('<H',d,n-2)[0]
            for i,col in enumerate(range(first_col,last_col+1)):
                rk=struct.unpack_from('<I',d,4+6*i+2)[0]
                rows[row][col]=rk_value(rk)
        elif code==0x0006 and n>=14: # FORMULA
            row,col,xf=struct.unpack_from('<HHH',d)
            val=struct.unpack_from('<d',d,6)[0]
            rows[row][col]=val
        if code==0x000A:break
        off+=4+n
    return rows


In [ ]:
"""Minimal corrections to the group's original ESE5420 notebook.
Use the same train/test split seed, LogisticRegression, fivefold CV and C grid.
Never fit on, predict on, or score the held-out test split in this script.
"""
from pathlib import Path
import json
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

HERE = Path.cwd()
C_GRID=[0.01,0.1,1.0,10.0]
SEED=42
COLS = ['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE',
        'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']+[f'BILL_AMT{i}' for i in range(1,7)]+[f'PAY_AMT{i}' for i in range(1,7)]


def fit_cv(X, y, cat_cols, num_cols, cv):
    preprocess = ColumnTransformer([
       ('scale_num',StandardScaler(),num_cols),
       ('encode_cat',OneHotEncoder(handle_unknown='ignore'),cat_cols),
    ])
    pipe=Pipeline([('preprocess',preprocess),('logistic',LogisticRegression(max_iter=2000,solver='lbfgs'))])
    search=GridSearchCV(estimator=pipe,param_grid={'logistic__C':C_GRID},
                        scoring={'log_loss':'neg_log_loss','roc_auc':'roc_auc'},
                        cv=cv,refit=False,n_jobs=2,error_score='raise')
    search.fit(X,y)
    res=search.cv_results_
    pos=int(np.argmax(res['mean_test_log_loss']))
    losses=np.array([-res[f'split{i}_test_log_loss'][pos] for i in range(5)])
    return {'best_C':float(res['param_logistic__C'][pos]),
            'fold_losses':losses.tolist(),
            'mean_cv_log_loss':float(np.mean(losses)),
            'fold_sd':float(np.std(losses,ddof=1)),
            'fold_se':float(np.std(losses,ddof=1)/np.sqrt(5)),
            'cv_roc_auc':float(res['mean_test_roc_auc'][pos])}

def main():
    rows=read_numeric_rows(str(HERE/'default of credit card clients.xls'))
    data=np.array([[rows[r][j] for j in range(25)] for r in sorted(rows)], dtype=np.float64)
    assert data.shape==(30000,25)
    assert np.array_equal(data[:,0],np.arange(1,30001))
    X=data[:,1:24];y=data[:,24].astype(int)
    assert np.isfinite(X).all() and set(np.unique(y))=={0,1}
    xtr,xtest,ytr,ytest=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
    base=fit_cv(xtr[:,0:5],ytr,[1,2,3],[0,4],cv)
    hist=fit_cv(xtr,ytr,[1,2,3,5,6,7,8,9,10],[0,4]+list(range(11,23)),cv)
    paired=np.array(base['fold_losses'])-np.array(hist['fold_losses'])
    report={'data_rows':int(len(y)),'predictors':int(X.shape[1]),'dropped':0,
            'defaults':int(y.sum()),'overall_default_rate':float(y.mean()),
            'train_rows':int(len(ytr)),'test_rows':int(len(ytest)),
            'train_defaults':int(ytr.sum()),'test_defaults':int(ytest.sum()),
            'train_default_rate':float(ytr.mean()),'test_default_rate':float(ytest.mean()),
            'split_seed':SEED,'cv_seed':SEED,'cv_folds':5,'c_grid':C_GRID,
            'baseline':base,'history':hist,
            'paired_mean_reduction':float(np.mean(paired)),
            'paired_fold_sd':float(np.std(paired,ddof=1)),
            'paired_fold_se':float(np.std(paired,ddof=1)/np.sqrt(5)),
            'paired_by_fold':paired.tolist(),
            'test_score_computed':False,
            'notes':'Original team notebook already displayed holdout test metrics. This corrected run performs CV only and does not score the holdout.'}
    (HERE/'project_corrected_metrics.json').write_text(json.dumps(report,indent=2))
    print('FEATURES',COLS)
    print(json.dumps(report,indent=2))
    return report
metrics = main()


## Interpreting the output

- `baseline`: LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE.
- `history`: the five baseline columns plus six each of PAY status, BILL_AMT, and PAY_AMT.
- **Paired mean reduction** is the mean across identical CV folds of baseline log loss minus history log loss (positive means history helps).
- Error bars are **descriptive standard errors over five folds**, not final test-set confidence intervals.
- The target is *default payment next month*. No held-out test metrics are computed in this notebook.
